In [2]:
from google.colab import drive
import pandas as pd
import numpy as np
import os

drive.mount('/content/drive')

PROJECT_PATH = "/content/drive/MyDrive/Store-Sales-Forecasting"
PROCESSED_PATH = f"{PROJECT_PATH}/Data/Processed"

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [3]:
df_model = pd.read_csv(
    f"{PROCESSED_PATH}/model_ready_dataset.csv",
    parse_dates=["date"]
)

print(df_model.shape)

(2975940, 31)


In [4]:
target = "sales"

drop_columns = [
    "id",
    "date",
    "sales",
    "holiday_type",
    "holiday_locale",
    "holiday_description"
]

X = df_model.drop(columns=drop_columns)
y = df_model[target]

categorical_columns = X.select_dtypes(include="object").columns.tolist()

X_encoded = pd.get_dummies(
    X,
    columns=categorical_columns,
    drop_first=True,
    dtype=int
)

print("Encoded features:", X_encoded.shape)

Encoded features: (2975940, 93)


In [5]:
cutoff_date = df_model["date"].max() - pd.Timedelta(days=90)

train_mask = df_model["date"] <= cutoff_date
test_mask = df_model["date"] > cutoff_date

X_train = X_encoded.loc[train_mask]
X_test = X_encoded.loc[test_mask]

y_train = y.loc[train_mask]
y_test = y.loc[test_mask]

print("Training rows:", X_train.shape[0])
print("Testing rows:", X_test.shape[0])

Training rows: 2815560
Testing rows: 160380


In [6]:
train_sample = pd.concat(
    [X_train, y_train.rename("sales")],
    axis=1
).sample(n=300000, random_state=42)

X_train_sample = train_sample.drop(columns="sales")
y_train_sample = train_sample["sales"]

print("Training sample:", X_train_sample.shape)

Training sample: (300000, 93)


XG BOOST

In [7]:
from xgboost import XGBRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np

xgb_model = XGBRegressor(
    n_estimators=300,
    learning_rate=0.05,
    max_depth=8,
    subsample=0.8,
    colsample_bytree=0.8,
    objective="reg:squarederror",
    random_state=42,
    n_jobs=-1
)

xgb_model.fit(X_train_sample, y_train_sample)

xgb_predictions = xgb_model.predict(X_test)

xgb_mae = mean_absolute_error(y_test, xgb_predictions)
xgb_rmse = np.sqrt(mean_squared_error(y_test, xgb_predictions))
xgb_r2 = r2_score(y_test, xgb_predictions)

print("XGBoost Results")
print("MAE:", round(xgb_mae, 2))
print("RMSE:", round(xgb_rmse, 2))
print("R²:", round(xgb_r2, 4))

XGBoost Results
MAE: 57.42
RMSE: 216.48
R²: 0.9735


In [8]:
results_df = pd.DataFrame({
    "Model": ["Random Forest", "XGBoost"],
    "MAE": [58.21, 57.42],
    "RMSE": [213.54, 216.48],
    "R2": [0.9742, 0.9735]
})

results_df = results_df.sort_values("RMSE").reset_index(drop=True)

print(results_df)

results_df.to_csv(
    f"{PROCESSED_PATH}/model_results.csv",
    index=False
)

print("Model results saved successfully.")

           Model    MAE    RMSE      R2
0  Random Forest  58.21  213.54  0.9742
1        XGBoost  57.42  216.48  0.9735
Model results saved successfully.


In [10]:
results_df = pd.DataFrame({
    "Model": ["Random Forest", "XGBoost"],
    "MAE": [58.21, 57.42],
    "RMSE": [213.54, 216.48],
    "R2": [0.9742, 0.9735]
})

results_df.to_csv(
    f"{PROCESSED_PATH}/model_results.csv",
    index=False
)

results_df

,Model,MAE,RMSE,R2
0,Random Forest,58.21,213.54,0.9742
1,XGBoost,57.42,216.48,0.9735
